In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(4048, 31)


In [2]:
product_features = (
    df.groupby("skuNumber")
    .agg(
        total_qty=(
            "effective_qty",
            "sum"
        ),

        total_orders=(
            "orderNumber",
            "nunique"
        ),

        unique_buyers=(
            "customerId",
            "nunique"
        ),

        avg_qty=(
            "effective_qty",
            "mean"
        )
    )
    .reset_index()
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty
0,SKU00001,27,23,23,1.173913
1,SKU00009,56,48,48,1.166667
2,SKU00010,155,111,110,1.396396
3,SKU00020,24,17,17,1.411765
4,SKU00022,95,86,84,1.104651


In [3]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName",
            "brand",
            "category",
            "subCategory"
        ]
    ]
    .drop_duplicates()
)

product_features = (
    product_features.merge(
        sku_lookup,
        on="skuNumber",
        how="left"
    )
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty,itemName,brand,category,subCategory
0,SKU00001,27,23,23,1.173913,Chingles Filz Jar,DS Group,Confectionery,Chewing Gum
1,SKU00009,56,48,48,1.166667,Chingles Maxi Tutti Fruiti Jar,DS Group,Confectionery,Chewing Gum
2,SKU00010,155,111,110,1.396396,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners,Mouth Freshener Mix
3,SKU00020,24,17,17,1.411765,Pass Pass FTT 19.2g MM,Pass Pass,Mouth Fresheners,Mouth Freshener Mix
4,SKU00022,95,86,84,1.104651,Pulse Kachha Aam 175 Candy,DS Group,Confectionery,Hard Boiled Candy


In [4]:
total_retailers = (
    df["customerId"]
    .nunique()
)

product_features[
    "popularity_score"
] = (
    product_features[
        "unique_buyers"
    ]
    / total_retailers
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty,itemName,brand,category,subCategory,popularity_score
0,SKU00001,27,23,23,1.173913,Chingles Filz Jar,DS Group,Confectionery,Chewing Gum,0.017037
1,SKU00009,56,48,48,1.166667,Chingles Maxi Tutti Fruiti Jar,DS Group,Confectionery,Chewing Gum,0.035556
2,SKU00010,155,111,110,1.396396,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners,Mouth Freshener Mix,0.081481
3,SKU00020,24,17,17,1.411765,Pass Pass FTT 19.2g MM,Pass Pass,Mouth Fresheners,Mouth Freshener Mix,0.012593
4,SKU00022,95,86,84,1.104651,Pulse Kachha Aam 175 Candy,DS Group,Confectionery,Hard Boiled Candy,0.062222


In [6]:
(
    product_features[
        [
            "itemName",
            "popularity_score"
        ]
    ]
    .sort_values(
        "popularity_score",
        ascending=False
    )
    .head(10)
)

,itemName,popularity_score
9,Rajnigandha 4g,0.196296
14,TZ 00 (with Silver) 0.45g,0.186667
10,RG Pearl Elaichi MP Pouch 0.14g,0.146667
153,Rajnigandha 17g Zipper 1 Pcs,0.137778
16,RG Pearl Elaichi 1.4g Hanger,0.104444
13,TZ 00 4.25g (6 Pcs),0.088889
18,Pass Pass Meetha Magic 2.2g (100 pcs),0.085185
2,Pass Pass Meetha Magic 11.75g Hanger,0.081481
54,Center Fruit,0.074074
4,Pulse Kachha Aam 175 Candy,0.062222


In [7]:
product_features[
    "category_rank"
] = (
    product_features
    .groupby("category")
    ["total_qty"]
    .rank(
        ascending=False,
        method="dense"
    )
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty,itemName,brand,category,subCategory,popularity_score,category_rank
0,SKU00001,27,23,23,1.173913,Chingles Filz Jar,DS Group,Confectionery,Chewing Gum,0.017037,11.0
1,SKU00009,56,48,48,1.166667,Chingles Maxi Tutti Fruiti Jar,DS Group,Confectionery,Chewing Gum,0.035556,6.0
2,SKU00010,155,111,110,1.396396,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners,Mouth Freshener Mix,0.081481,4.0
3,SKU00020,24,17,17,1.411765,Pass Pass FTT 19.2g MM,Pass Pass,Mouth Fresheners,Mouth Freshener Mix,0.012593,7.0
4,SKU00022,95,86,84,1.104651,Pulse Kachha Aam 175 Candy,DS Group,Confectionery,Hard Boiled Candy,0.062222,2.0


In [8]:
product_features.to_parquet(
    "../data/processed/product_features.parquet",
    index=False
)

print("Saved")

Saved
